# Display Panel ROI Processing

이 노트북은 mono camera로 촬영된 display panel 이미지를 처리합니다.

## 처리 순서:
1. 이미지 로드 및 기본 정보 확인
2. ROI (디스플레이 영역) 검출
3. 디스플레이 경계 검출 및 코너 추출
4. Warp를 통한 tilt 보정
5. 디스플레이 해상도로 리사이즈
6. 16bit 정규화 및 최종 결과 저장

## Step 1: 이미지 로드 및 기본 정보 확인

In [ ]:
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
import os

# 작업 디렉토리 설정
work_dir = Path(r'c:\Users\byungpaul\Desktop\AI_Project\20260304_ROI_algorithm')
data_dir = work_dir / 'data'
output_dir = work_dir / 'output'
output_dir.mkdir(exist_ok=True)

# 이미지 로드
img_path = data_dir / 'G32_cal.tif'
img = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)

print(f"이미지 경로: {img_path}")
print(f"이미지 shape: {img.shape}")
print(f"이미지 dtype: {img.dtype}")
print(f"이미지 min: {img.min()}, max: {img.max()}")
print(f"이미지 mean: {img.mean():.2f}")

# 이미지 히스토그램 확인
plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title(f'Original Image ({img.shape[1]}x{img.shape[0]})')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.hist(img.ravel(), bins=256, range=(0, img.max()), color='gray', alpha=0.7)
plt.title('Histogram')
plt.xlabel('Pixel Value')
plt.ylabel('Frequency')
plt.yscale('log')

plt.tight_layout()
plt.savefig(output_dir / '01_original_image.png', dpi=150)
plt.show()

## Step 2: ROI (디스플레이 영역) 검출

In [ ]:
# ROI Detector 모듈 추가
import sys
sys.path.insert(0, str(work_dir / 'src'))

from roi_detector import ROIDetector

# ROI Detector 초기화
detector = ROIDetector(debug=True)

# 이미지 설정 및 ROI 검출
detector.set_image(img)

# 초기 ROI 검출 (디스플레이 영역 대략적 위치)
initial_roi = detector.detect_initial_roi()
print(f"\n초기 ROI 검출 결과: {initial_roi}")

# 초기 ROI 시각화
if initial_roi:
    x, y, w, h = initial_roi
    plt.figure(figsize=(12, 8))

    # 8비트 변환 (시각화용)
    img_8bit = ((img - img.min()) / (img.max() - img.min()) * 255).astype(np.uint8)
    img_color = cv2.cvtColor(img_8bit, cv2.COLOR_GRAY2BGR)

    # ROI 영역 표시
    cv2.rectangle(img_color, (x, y), (x + w, y + h), (0, 255, 0), 5)

    plt.imshow(cv2.cvtColor(img_color, cv2.COLOR_BGR2RGB))
    plt.title(f'Initial ROI Detection (x={x}, y={y}, w={w}, h={h})')
    plt.savefig(output_dir / '02_initial_roi.png', dpi=150)
    plt.show()
else:
    print("ROI 검출 실패")

## Step 3: 디스플레이 경계 검출 및 코너 추출

In [ ]:
# 코너 검출 (4개의 코너 포인트)
corners = detector.detect_corners()

print("\n검출된 코너 좌표:")
for name, point in corners.items():
    print(f"  {name}: ({point[0]:.1f}, {point[1]:.1f})")

# 코너 시각화
plt.figure(figsize=(15, 10))

# 전체 이미지에 코너 표시
img_corners = cv2.cvtColor(img_8bit.copy(), cv2.COLOR_GRAY2BGR)

corner_colors = {
    'top_left': (255, 0, 0),      # Blue
    'top_right': (0, 255, 0),     # Green
    'bottom_left': (0, 0, 255),   # Red
    'bottom_right': (255, 255, 0) # Cyan
}

for name, point in corners.items():
    pt = (int(point[0]), int(point[1]))
    cv2.circle(img_corners, pt, 20, corner_colors[name], -1)
    cv2.putText(img_corners, name, (pt[0] + 30, pt[1]),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, corner_colors[name], 3)

# 코너를 연결하는 사각형
pts = np.array([
    corners['top_left'],
    corners['top_right'],
    corners['bottom_right'],
    corners['bottom_left']
], dtype=np.int32)
cv2.polylines(img_corners, [pts], True, (0, 255, 255), 3)

plt.subplot(1, 2, 1)
plt.imshow(cv2.cvtColor(img_corners, cv2.COLOR_BGR2RGB))
plt.title('Detected Corners')

# 검출된 영역 확대
plt.subplot(1, 2, 2)
min_x = int(min(pt[0] for pt in corners.values())) - 50
max_x = int(max(pt[0] for pt in corners.values())) + 50
min_y = int(min(pt[1] for pt in corners.values())) - 50
max_y = int(max(pt[1] for pt in corners.values())) + 50

plt.imshow(cv2.cvtColor(img_corners[max(0,min_y):max_y, max(0,min_x):max_x], cv2.COLOR_BGR2RGB))
plt.title('Zoomed Corner Region')

plt.tight_layout()
plt.savefig(output_dir / '03_corners.png', dpi=150)
plt.show()

## Step 4: Warp를 통한 Tilt 보정

In [ ]:
# 디스플레이 해상도 설정 (예: Full HD)
# TODO: 실제 디스플레이 해상도에 맞게 조정 필요
display_width = 2160   # 디스플레이 가로 해상도
display_height = 2160  # 디스플레이 세로 해상도

# 소스 포인트 (검출된 코너)
src_pts = np.array([
    corners['top_left'],
    corners['top_right'],
    corners['bottom_right'],
    corners['bottom_left']
], dtype=np.float32)

# 목적지 포인트 (직사각형)
dst_pts = np.array([
    [0, 0],
    [display_width - 1, 0],
    [display_width - 1, display_height - 1],
    [0, display_height - 1]
], dtype=np.float32)

# Perspective Transform Matrix 계산
M = cv2.getPerspectiveTransform(src_pts, dst_pts)

print("Perspective Transform Matrix:")
print(M)

# Warp 적용 (16비트 유지)
warped = cv2.warpPerspective(img, M, (display_width, display_height),
                              flags=cv2.INTER_LINEAR)

print(f"\n원본 이미지 shape: {img.shape}")
print(f"Warped 이미지 shape: {warped.shape}")
print(f"Warped 이미지 dtype: {warped.dtype}")
print(f"Warped min: {warped.min()}, max: {warped.max()}")

# Warp 결과 시각화
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title(f'Original ({img.shape[1]}x{img.shape[0]})')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(warped, cmap='gray')
plt.title(f'Warped ({warped.shape[1]}x{warped.shape[0]})')
plt.colorbar()

plt.tight_layout()
plt.savefig(output_dir / '04_warped.png', dpi=150)
plt.show()

## Step 5: 디스플레이 해상도로 리사이즈

In [ ]:
# Warp된 이미지가 이미 목표 해상도이므로 추가 리사이즈 불필요
# 필요한 경우 다른 해상도로 리사이즈

final_width = display_width
final_height = display_height

# 이미 목표 해상도인 경우 그대로 사용
if warped.shape[1] == final_width and warped.shape[0] == final_height:
    resized = warped.copy()
    print(f"이미 목표 해상도입니다: {final_width}x{final_height}")
else:
    resized = cv2.resize(warped, (final_width, final_height),
                          interpolation=cv2.INTER_LINEAR)
    print(f"리사이즈 완료: {warped.shape[1]}x{warped.shape[0]} -> {final_width}x{final_height}")

print(f"\n최종 이미지 shape: {resized.shape}")
print(f"최종 이미지 dtype: {resized.dtype}")

## Step 6: 16bit 정규화 및 최종 결과 저장

In [ ]:
# 16bit 정규화 (0-65535 범위로)
resized_min = resized.min()
resized_max = resized.max()

if resized_max > resized_min:
    normalized = ((resized - resized_min) / (resized_max - resized_min) * 65535).astype(np.uint16)
else:
    normalized = np.zeros_like(resized, dtype=np.uint16)

print(f"정규화 전 범위: [{resized_min}, {resized_max}]")
print(f"정규화 후 범위: [{normalized.min()}, {normalized.max()}]")
print(f"정규화된 이미지 dtype: {normalized.dtype}")

# 결과 저장
output_filename = 'G32_cal_processed.tif'
output_path = output_dir / output_filename

# 16bit TIFF로 저장
from PIL import Image
Image.fromarray(normalized).save(str(output_path))
print(f"\n결과 저장 완료: {output_path}")

# 최종 결과 시각화
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.imshow(img, cmap='gray')
plt.title(f'Original Image\n{img.shape[1]}x{img.shape[0]}, range: [{img.min()}, {img.max()}]')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(normalized, cmap='gray')
plt.title(f'Processed Result\n{normalized.shape[1]}x{normalized.shape[0]}, range: [0, 65535]')
plt.colorbar()

plt.tight_layout()
plt.savefig(output_dir / '05_final_result.png', dpi=150)
plt.show()

print("\n===== 처리 완료 =====")
print(f"입력: {img_path}")
print(f"출력: {output_path}")
print(f"원본 크기: {img.shape[1]}x{img.shape[0]}")
print(f"출력 크기: {normalized.shape[1]}x{normalized.shape[0]}")
print(f"출력 포맷: 16-bit TIFF")